<table>
<tr><td><img style="height: 150px;" src="images/geo_hydro1.jpg"></td>
<td bgcolor="#FFFFFF">
    <p style="font-size: xx-large; font-weight: 900; line-height: 100%">AG Dynamics of the Earth</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Juypter notebooks</p>
    <p style="font-size: large; color: rgba(0,0,0,0.5);">Georg Kaufmann</p>
    </td>
</tr>
</table>

# Machine learning: Feature reduction

----
*Georg Kaufmann,
Geophysics Section,
Institute of Geological Sciences,
Freie Universität Berlin,
Germany*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap 
import csv

### Load iris data set

We first load our cleaned iris data set again, and plot one more time the different feature correlations.

In [ ]:
n=150
m=4
X = np.zeros([n*m]).reshape(n,m)
y = np.zeros([n])
yname=np.empty([n],dtype='object')

csvfile = open('data/iris.csv', newline='')
iris = csv.reader(csvfile, delimiter=',')
next(iris)
i=0
for row in iris:
    X[i,0] = row[0]
    X[i,1] = row[1]
    X[i,2] = row[2]
    X[i,3] = row[3]
    yname[i]=row[4]
    if (row[4]=='Iris-setosa'):
        y[i] = 0
    elif (row[4]=='Iris-versicolor'):
        y[i] = 1
    elif (row[4]=='Iris-virginica'):
        y[i] = 2
    i=i+1
print(X.ndim,X.shape,X[0])
print(y.ndim,y.shape,y[0])
print(yname.ndim,yname.shape)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10,5), sharey=False)
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
#axs[0].set_xlim([2,10])
#axs[0].set_ylim([1,5])
axs[0].set_xlabel('Sepal length')
axs[0].set_ylabel('Sepal width')
axs[0].scatter(X[:,0],X[:,1],c=y,cmap=cmap_bold)
#axs[1].set_xlim([2,10])
#axs[1].set_ylim([1,5])
axs[1].set_xlabel('Petal length')
axs[1].set_ylabel('Petal width')
axs[1].scatter(X[:,2],X[:,3],c=y,cmap=cmap_bold)

## Feature analysis

Our question is: Can we analyse the four features to figure out, which ones are **better suited** for
the analysis.

We use the **principal component analysis** for this task.

A simple example for explanation. We have **two** features, which seem to be highly correlated.

In [ ]:
n=100
a=0.
b=1.
Xtest = np.zeros(2*n).reshape(n,2)
Xtest[:,0] = np.linspace(-2,2,n)
Xtest[:,1] = a + b*Xtest[:,0] + np.random.uniform(-0.4,0.4,n)
plt.figure(figsize=(10,6))
plt.scatter(Xtest[:,0],Xtest[:,1])

Next, we use the `PCA` module from the `scikit-learn` package to perform a
*principal component analysis**:

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(Xtest)

In [ ]:
print(pca.components_)
print(pca.explained_variance_)

In [ ]:
def draw_vector(v0, v1, ax=None):
    ax = ax or plt.gca()
    arrowprops=dict(arrowstyle='->',
                    linewidth=2,
                    shrinkA=0, shrinkB=0)
    ax.annotate('', v1, v0, arrowprops=arrowprops)

# plot data
plt.figure(figsize=(10,6))
plt.scatter(Xtest[:, 0], Xtest[:, 1], alpha=0.2)
for length, vector in zip(pca.explained_variance_, pca.components_):
    v = vector * 1 * np.sqrt(length)
    draw_vector(pca.mean_, pca.mean_ + v)
plt.axis('equal');

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fig.subplots_adjust(left=0.0625, right=0.95, wspace=0.1)

# plot data
ax[0].scatter(Xtest[:, 0], Xtest[:, 1], alpha=0.2)
for length, vector in zip(pca.explained_variance_, pca.components_):
    v = vector * 1 * np.sqrt(length)
    draw_vector(pca.mean_, pca.mean_ + v, ax=ax[0])
ax[0].axis('equal');
ax[0].set(xlabel='x', ylabel='y', title='input')

# plot principal components
X_pca = pca.transform(Xtest)
ax[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.2)
draw_vector([0, 0], [0, 3], ax=ax[1])
draw_vector([0, 0], [3, 0], ax=ax[1])
ax[1].axis('equal')
ax[1].set(xlabel='component 1', ylabel='component 2',
          title='principal components',
          xlim=(-5, 5), ylim=(-3, 3.1))

----
## Iris example

For the iris example, we use the **four** features, but want to condense them to the **two**
most important ones.

In [ ]:
from sklearn.decomposition import PCA
print(X.shape)
pca = PCA(n_components=2, whiten=True)
pca.fit(X)
Xpca = pca.transform(X)
print(Xpca.shape)

In [ ]:
plt.figure(figsize=(8,8))
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
#axs[0].set_xlim([2,10])
#axs[0].set_ylim([1,5])
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.scatter(Xpca[:,0],Xpca[:,1],c=y,cmap=cmap_bold)

Now we use these two new features for prediction.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Xpca, y, test_size=0.4, random_state=42, stratify=y)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])
cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
font_size=20

def meshGrid(x_data, y_data):
    h = .02  # step size in the mesh
    x_min, x_max = x_data.min() - 1, x_data.max() + 1
    y_min, y_max = y_data.min() - 1, y_data.max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    return (xx,yy)
    
def plotPrediction(clf,x_train,y_train,x_test,y_test,x_label,y_label,col_train,col_test, title="", mesh=True):
    xx_train,yy_train = meshGrid(x_train, y_train)
    Z_train = clf.predict(np.c_[xx_train.ravel(), yy_train.ravel()])
    xx_test,yy_test = meshGrid(x_test, y_test)
    Z_test = clf.predict(np.c_[xx_test.ravel(), yy_test.ravel()])

    # Put the result into a color plot
    Z_train = Z_train.reshape(xx_train.shape)
    Z_test  = Z_test.reshape(xx_test.shape)
    
    fig, axs = plt.subplots(1, 2, figsize=(14,7), sharey=False)
    if mesh:
        axs[0].pcolormesh(xx_train, yy_train, Z_train, cmap=cmap_light)
    axs[0].set_xlim(xx_train.min(), xx_train.max())
    axs[0].set_ylim(yy_train.min(), yy_train.max())
    axs[0].set_xlabel(x_label, fontsize=font_size)
    axs[0].set_ylabel(y_label, fontsize=font_size)
    axs[0].set_title('train data', fontsize=font_size)
    axs[0].scatter(x_train, y_train, c=col_train, cmap=cmap_bold, s=80, marker='o')

    if mesh:
        axs[1].pcolormesh(xx_test, yy_test, Z_test, cmap=cmap_light)
    axs[1].set_xlim(xx_test.min(), xx_test.max())
    axs[1].set_ylim(yy_test.min(), yy_test.max())
    axs[1].set_xlabel(x_label, fontsize=font_size)
    axs[1].set_ylabel(y_label, fontsize=font_size)
    axs[1].set_title('test data', fontsize=font_size)
    axs[1].scatter(x_test, y_test, c=col_test, cmap=cmap_bold, s=80, marker='o')

    fig.suptitle(title, fontsize=font_size)

In [ ]:
nn=10
from sklearn import neighbors
clf_sepal = neighbors.KNeighborsClassifier(n_neighbors=nn)
clf_sepal.fit(X_train, y_train)

print('Score fit train data: ',clf_sepal.score(X_train, y_train))
print('Score fit test data:  ',clf_sepal.score(X_test, y_test))
plotPrediction(clf_sepal, 
               X_train[:, 0], X_train[:, 1],
               X_test[:, 0], X_test[:, 1],
               'Feature 1', 'Feature 2', y_train,y_test, mesh=True,
                title="Data after PCA (n="+str(nn)+")")

... done